In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/train_prepared.csv")
print(df.shape)

(1290081, 12)


In [22]:
df['origin_inconsistency'] = abs(
    (df['origin_balance_before'] - df['amount']) - df['origin_balance_after']
)

df['dest_inconsistency'] = abs(
    (df['destination_balance_before'] + df['amount']) - df['destination_balance_after']
)

print(df[['origin_inconsistency', 'dest_inconsistency']].describe())

       origin_inconsistency  dest_inconsistency
count          1.290081e+06        1.290081e+06
mean           7.802839e+04        4.793052e+04
std            1.656009e+05        8.853031e+04
min            0.000000e+00        0.000000e+00
25%            0.000000e+00        0.000000e+00
50%            1.000000e-02        1.000000e-02
75%            1.233700e+03        6.600736e+04
max            1.717166e+06        1.717166e+06


In [23]:
df.groupby('fraud_flag')[['origin_inconsistency', 'dest_inconsistency']].mean()

,origin_inconsistency,dest_inconsistency
fraud_flag,,
0,86420.617039,52963.157357
1,2844.197747,2844.196153


In [24]:
df['origin_account_emptied'] = (df['origin_balance_after'] == 0).astype(int)
df['amount_to_balance_ratio'] = df['amount'] / (df['origin_balance_before'] + 1)
df['balance_diff_dest'] = df['destination_balance_after'] - df['destination_balance_before']
df['is_op03'] = (df['operation'] == 'op_03').astype(int)
df['surge_origin'] = df.groupby(['origin_account', 'period'])['id'].transform('count')
df['surge_dest'] = df.groupby(['destination_account', 'period'])['id'].transform('count')
df['surge_origin'] = df.groupby(['origin_account', 'period'])['id'].transform('count')
df['surge_dest']   = df.groupby(['destination_account', 'period'])['id'].transform('count')
amount_mean = df.groupby('operation')['amount'].transform('mean')
df['amount_vs_op_mean'] = df['amount'] / (amount_mean + 1)

print(df[['origin_account_emptied', 'amount_to_balance_ratio', 'balance_diff_dest', 'is_op03']].describe())

       origin_account_emptied  amount_to_balance_ratio  balance_diff_dest  \
count            1.290081e+06             1.290081e+06       1.290081e+06   
mean             5.511282e-03             3.054234e+01       1.622987e+04   
std              7.403318e-02             1.117598e+03       6.971475e+04   
min              0.000000e+00            -1.539029e+03       0.000000e+00   
25%              0.000000e+00             3.372162e-04       0.000000e+00   
50%              0.000000e+00             7.186208e-03       6.939700e+02   
75%              0.000000e+00             4.198025e-02       2.005356e+04   
max              1.000000e+00             2.901223e+05       2.526958e+06   

            is_op03  
count  1.290081e+06  
mean   3.219356e-01  
std    4.672186e-01  
min    0.000000e+00  
25%    0.000000e+00  
50%    0.000000e+00  
75%    1.000000e+00  
max    1.000000e+00  


In [25]:
df.groupby('fraud_flag')[['origin_account_emptied', 'amount_to_balance_ratio', 'balance_diff_dest']].mean()

,origin_account_emptied,amount_to_balance_ratio,balance_diff_dest
fraud_flag,,,
0,0.006126,33.859903,15005.531519
1,0.000000,0.821019,27198.408158


In [26]:
# Label encoding simple
operation_mapping = {op: i for i, op in enumerate(df['operation'].unique())}
df['operation_encoded'] = df['operation'].map(operation_mapping)

print(operation_mapping)

{'op_05': 0, 'op_03': 1, 'op_04': 2, 'op_02': 3, 'op_01': 4}


In [27]:
features = [
    'period', 'amount', 'is_op03', 'operation_encoded',
    'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'balance_diff_origin', 'balance_diff_dest',
    'origin_inconsistency', 'dest_inconsistency',
    'origin_account_emptied', 'amount_to_balance_ratio',
    'surge_origin', 'surge_dest', 'amount_vs_op_mean',
]

X = df[features]
y = df['fraud_flag']

print(f"Features : {X.shape[1]}")
print(f"Exemples : {X.shape[0]}")
print(f"Taux de fraude : {y.mean():.4f}")

Features : 17
Exemples : 1290081
Taux de fraude : 0.1004


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"Taux fraude train : {y_train.mean():.4f}")
print(f"Taux fraude val : {y_val.mean():.4f}")

X_train: (1032064, 17)
X_val: (258017, 17)
Taux fraude train : 0.1004
Taux fraude val : 0.1004


In [29]:
fraud_count = y_train.sum()
normal_count = len(y_train) - fraud_count
scale_pos_weight = normal_count / fraud_count

print(f"Transactions normales  : {normal_count:,}")
print(f"Transactions frauduleuses : {fraud_count:,}")
print(f"scale_pos_weight (pour XGBoost) : {scale_pos_weight:.2f}")

Transactions normales  : 928,430
Transactions frauduleuses : 103,634
scale_pos_weight (pour XGBoost) : 8.96


In [30]:
df.to_csv("../data/processed/train_featured.csv", index=False)
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_val.to_csv("../data/processed/X_val.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)

print("Fichiers sauvegardés.")

Fichiers sauvegardés.
